In [0]:
SELECT * FROM dev.mohit_gangwani.2k_first_enabled_set

-- WHERE vendor_name = 'TMS'

In [0]:
SELECT a.*
FROM dev.ltao_dev.data2419_full_0825 a
LEFT JOIN stage.detection.inscape_station_map ism
ON a.station_call_sign = ism.mapped_vendor_call_sign
AND a.vendor_name = ism.mapped_vendor
LEFT JOIN dev.mohit_gangwani.2k_first_enabled_set b
  ON a.station_call_sign = b.station_call_sign
WHERE b.station_call_sign IS NULL

In [0]:
SELECT * FROM prod.detection.inscape_station_map
WHERE mapped_vendor_call_sign LIKE 'KAKWD%'

In [0]:

-- DROP TABLE IF EXISTS dev.mohit_gangwani.2k_first_enabled_set;
-- CREATE TABLE dev.mohit_gangwani.2k_first_enabled_set AS
SELECT a.station_id, a.station_num, a.station_name, a.station_call_sign, a.station_affil, a.fk_dma_id, a.inscape_station_name
FROM dev.ltao_dev.data2419_full_0825 a
JOIN stage.detection.epg_station st
ON a.station_call_sign = st.station_call_sign
AND a.vendor_name= st.vendor_name
-- WHERE a.vendor_name= 'TIVO'
AND st.attributed = 'TRUE'
AND st.station_num = 186192
GROUP BY ALL;

In [0]:
SELECT * FROM stage.detection.inscape_station_map
WHERE inscape_call_sign = 'KMBYLD5'

In [0]:
SELECT COUNT(*) FROM dev.mohit_gangwani.2k_first_enabled_set

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.ny_nodma_stations;
CREATE TABLE dev.mohit_gangwani.ny_nodma_stations AS
SELECT *
FROM stage.detection.epg_station
WHERE station_call_sign IN ('KJWPDT', 'WABCDT', 'WABCDT2', 'WCBSDT', 'WCBSDT2', 'WCBSDT3', 'WCBSDT4', 'WFUTDT', 'WFUTDT2', 'WFUTDT3', 'WGENDT', 'WJLPDT', 'WJLPDT2', 'WJLPDT3', 'WJLPDT4', 'WLNYDT2', 'WMCNDT2', 'WNBCDT', 'WNBCDT2', 'WNETDT', 'WNJUDT', 'WNYWDT', 'WNYWDT2', 'WNYWDT5', 'WPIXDT', 'WPIXDT2', 'WPIXDT3', 'WPXNDT', 'WPXNDT4', 'WPXNDT6', 'WWORDT', 'WXTVDT', 'WXTVDT2')
AND vendor_name = 'TIVO'

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.ingested_stations_schedule_091525;
CREATE TABLE dev.mohit_gangwani.ingested_stations_schedule_091525 AS
SELECT sch.airdate, st.station_call_sign, sh.database_key, sch.airdate + INTERVAL '168 HOURS' AS end_of_ingestion
FROM prod.detection.epg_schedule sch
JOIN prod.detection.epg_station st
  ON st.station_id = sch.fk_station_id
 AND st.vendor_name = sch.vendor_name
JOIN prod.detection.epg_show sh
  ON sh.show_id = sch.fk_show_id
 AND sh.vendor_name = sch.vendor_name
WHERE st.ingested = 'TRUE'
  AND sch.vendor_name = 'TIVO'
  AND sch.airdate > CURRENT_DATE - interval '8 days'
  AND sch.airdate < CURRENT_DATE + interval '8 days'
GROUP BY ALL;

In [0]:
SELECT * FROM dev.mohit_gangwani.2k_first_enabled_set

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.fist_2k_unique_ingested_shows;
CREATE TABLE dev.mohit_gangwani.fist_2k_unique_ingested_shows AS
WITH existing_stations_in_the_same_dma AS (
  SELECT st.station_id, st.station_name, st.station_call_sign, st.station_affil, st.fk_dma_id, st.inscape_station_name
  FROM stage.detection.epg_station st
  JOIN dev.mohit_gangwani.2k_first_enabled_set a
    ON a.station_call_sign != st.station_call_sign
   AND a.fk_dma_id <=> st.fk_dma_id
  WHERE st.vendor_name = 'TIVO'
    AND (st.attributed = 'TRUE' OR st.ingested = 'TRUE')
    AND st.fk_dma_id IS NOT NULL
    AND st.station_call_sign NOT IN (SELECT station_call_sign FROM dev.mohit_gangwani.2k_first_enabled_set)
  GROUP BY ALL
)
, existing_ingest_schedule AS (
  SELECT sch.airdate, st.station_call_sign, sh.database_key, st.fk_dma_id
  FROM stage.detection.epg_schedule sch
  JOIN existing_stations_in_the_same_dma st
    ON st.station_id = sch.fk_station_id
  JOIN stage.detection.epg_show sh
    ON sh.show_id = sch.fk_show_id
   AND sh.vendor_name = sch.vendor_name
  JOIN dev.mohit_gangwani.ingested_stations_schedule_091525 ing_sch
    ON ing_sch.database_key = sh.database_key
   AND sch.airdate >= ing_sch.airdate
   AND sch.airdate <= ing_sch.end_of_ingestion
  WHERE sch.airdate >= CURRENT_DATE - 7
    AND sch.airdate < CURRENT_DATE + 1
    AND sch.vendor_name = 'TIVO'
  GROUP BY ALL
)
, new_stations_ingested_schedule AS (
  SELECT sch.airdate, st.station_call_sign, sh.database_key, st.fk_dma_id
  FROM stage.detection.epg_schedule sch
  JOIN dev.mohit_gangwani.2k_first_enabled_set st
    ON st.station_id = sch.fk_station_id
  JOIN stage.detection.epg_show sh
    ON sh.show_id = sch.fk_show_id
   AND sh.vendor_name = sch.vendor_name
  JOIN dev.mohit_gangwani.ingested_stations_schedule_091525 ing_sch
    ON ing_sch.database_key = sh.database_key
   AND sch.airdate >= ing_sch.airdate
   AND sch.airdate <= ing_sch.end_of_ingestion
  WHERE sch.airdate >= CURRENT_DATE - 7
    AND sch.airdate < CURRENT_DATE
    AND sch.vendor_name = 'TIVO'
  GROUP BY ALL
)
SELECT station_call_sign, database_key, airdate, fk_dma_id
FROM (
SELECT n.station_call_sign, n.database_key, n.airdate, n.fk_dma_id, COUNT(DISTINCT o.database_key||'_'||o.airdate) AS in_existing_stations
FROM new_stations_ingested_schedule n
LEFT JOIN existing_ingest_schedule o
  ON n.fk_dma_id = o.fk_dma_id
 AND n.airdate = o.airdate
 AND n.database_key = o.database_key
GROUP BY 1, 2, 3, 4)
WHERE in_existing_stations = 0

In [0]:
SELECT station_call_sign, COUNT(DISTINCT database_key) AS total_shows, COUNT(DISTINCT database_key||'_'||airdate) AS total_airings
FROM dev.mohit_gangwani.fist_2k_unique_ingested_shows
GROUP BY 1
ORDER BY 3 DESC, 2 DESC

In [0]:
-- SELECT station_call_sign, database_key, airdate
WITH no_viewing_airings AS (
  SELECT station_call_sign, fk_dma_id, COUNT(DISTINCT database_key) AS missed_shows, COUNT(DISTINCT database_key||'_'||airdate) AS missed_airings
  FROM (
    SELECT sch.station_call_sign, sch.database_key, sch.airdate, sch.fk_dma_id, COUNT(DISTINCT fk_tvid||'_'||session_start) AS session_count, COUNT(DISTINCT fk_tvid) AS tv_count
    FROM dev.mohit_gangwani.fist_2k_unique_ingested_shows sch
    JOIN stage.detection.epg_station st
      ON st.station_call_sign = sch.station_call_sign
    AND st.vendor_name = 'TIVO'
    JOIN stage.detection.epg_show sh
      ON sh.database_key = sch.database_key
    AND sh.vendor_name = 'TIVO'
    LEFT JOIN stage.detection.viewing_content_firehose vc
      ON vc.airdate = sch.airdate
    AND sh.show_id = vc.fk_show_id
    AND st.station_id = vc.fk_station_id
    AND vc.session_start >= CURRENT_DATE - 7
    AND vc.session_duration > 0
    GROUP BY 1,2,3,4
  )
  WHERE session_count = 0
  GROUP BY 1, 2
  -- ORDER BY 3 DESC, 2 DESC, 1
)
, all_airings AS (
  SELECT station_call_sign, fk_dma_id, COUNT(DISTINCT database_key) AS total_shows, COUNT(DISTINCT database_key||'_'||airdate) AS total_airings
  FROM dev.mohit_gangwani.fist_2k_unique_ingested_shows
  GROUP BY 1, 2
  -- ORDER BY 3 DESC, 2 DESC
)
SELECT a.station_call_sign, a.fk_dma_id, a.total_airings, n.missed_airings, n.missed_airings*1.0/a.total_airings AS perc_missed
FROM all_airings a
LEFT JOIN no_viewing_airings n
  ON a.station_call_sign = n.station_call_sign
ORDER BY 4

In [0]:
    SELECT COUNT(DISTINCT fk_tvid||'_'||session_start) AS session_count, COUNT(DISTINCT fk_tvid) AS tv_count
    FROM dev.mohit_gangwani.fist_2k_unique_ingested_shows sch
    JOIN stage.detection.epg_station st
      ON st.station_call_sign = sch.station_call_sign
    AND st.vendor_name = 'TIVO'
    JOIN stage.detection.epg_show sh
      ON sh.database_key = sch.database_key
    AND sh.vendor_name = 'TIVO'
    JOIN stage.detection.viewing_content_firehose vc
      ON vc.airdate = sch.airdate
      AND sh.show_id = vc.fk_show_id
      AND st.station_id = vc.fk_station_id
      AND vc.session_start >= CURRENT_DATE - 7
      AND vc.session_duration > 0
    -- GROUP BY 1,2,3,4

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.missed_ingested_airings_on_2k_stations;
CREATE TABLE dev.mohit_gangwani.missed_ingested_airings_on_2k_stations AS
SELECT station_call_sign, fk_dma_id, database_key, airdate
FROM (
  SELECT sch.station_call_sign, sch.database_key, sch.airdate, sch.fk_dma_id, COUNT(DISTINCT fk_tvid||'_'||session_start) AS session_count, COUNT(DISTINCT fk_tvid) AS tv_count
  FROM dev.mohit_gangwani.fist_2k_unique_ingested_shows sch
  JOIN stage.detection.epg_station st
    ON st.station_call_sign = sch.station_call_sign
  AND st.vendor_name = 'TIVO'
  JOIN stage.detection.epg_show sh
    ON sh.database_key = sch.database_key
  AND sh.vendor_name = 'TIVO'
  LEFT JOIN stage.detection.viewing_content_firehose vc
    ON vc.airdate = sch.airdate
  AND sh.show_id = vc.fk_show_id
  AND st.station_id = vc.fk_station_id
  AND vc.session_start >= CURRENT_DATE - 7
  AND vc.session_duration > 0
  GROUP BY 1,2,3,4
)
WHERE session_count = 0
GROUP BY ALL;

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.viewing_on_other_stations_for_missed_ingested_airings_on_new_2k_stations;
CREATE TABLE dev.mohit_gangwani.viewing_on_other_stations_for_missed_ingested_airings_on_new_2k_stations AS
SELECT st.station_call_sign, st.fk_dma_id, vc.airdate, sh.database_key, COUNT(DISTINCT fk_tvid||'_'||session_start) AS session_count
FROM stage.detection.viewing_content_firehose vc
JOIN stage.detection.epg_show sh
  ON sh.show_id = vc.fk_show_id
 AND sh.vendor_name = 'TIVO'
JOIN stage.detection.epg_station st
  ON st.station_id = vc.fk_station_id
 AND st.vendor_name = 'TIVO'
JOIN dev.mohit_gangwani.missed_ingested_airings_on_2k_stations miss
  ON miss.airdate = vc.airdate
 AND miss.database_key = sh.database_key
 AND miss.fk_dma_id = st.fk_dma_id
 AND miss.station_call_sign != st.station_call_sign
WHERE vc.session_start >= CURRENT_DATE - 7
  AND vc.session_start < CURRENT_DATE
GROUP BY 1, 2, 3, 4

In [0]:
SELECT * FROM dev.mohit_gangwani.viewing_on_other_stations_for_missed_ingested_airings_on_new_2k_stations

In [0]:
SELECT * FROM stage.detection.epg_show
WHERE database_key IN ('825841' ,'15316770473' ,'12406912733' ,'1915752' ,'16143537257' ,'15928565757')

In [0]:
SELECT DATE(vc.session_start) AS viewing_date, sch.station_call_sign, COUNT(DISTINCT fk_tvid||'_'||session_start) AS session_count, COUNT(DISTINCT fk_tvid) AS tv_count
FROM dev.mohit_gangwani.fist_2k_unique_ingested_shows sch
JOIN stage.detection.epg_station st
  ON st.station_call_sign = sch.station_call_sign
AND st.vendor_name = 'TIVO'
JOIN stage.detection.epg_show sh
  ON sh.database_key = sch.database_key
AND sh.vendor_name = 'TIVO'
LEFT JOIN stage.detection.viewing_content_firehose vc
  ON vc.airdate = sch.airdate
AND sh.show_id = vc.fk_show_id
AND st.station_id = vc.fk_station_id
AND vc.session_start >= CURRENT_DATE - 7
AND vc.session_duration > 0
GROUP BY 1,2

Databricks visualization. Run in Databricks to view.

In [0]:
-- SELECT station_call_sign, database_key, airdate
WITH no_viewing_airings AS (
  SELECT station_call_sign, COUNT(DISTINCT database_key) AS missed_shows, COUNT(DISTINCT database_key||'_'||airdate) AS missed_airings
  FROM (
    SELECT sch.station_call_sign, sch.database_key, sch.airdate, COUNT(DISTINCT fk_tvid||'_'||session_start) AS session_count, COUNT(DISTINCT fk_tvid) AS tv_count
    FROM dev.mohit_gangwani.fist_2k_unique_ingested_shows sch
    JOIN stage.detection.epg_station st
      ON st.station_call_sign = sch.station_call_sign
    AND st.vendor_name = 'TIVO'
    JOIN stage.detection.epg_show sh
      ON sh.database_key = sch.database_key
    AND sh.vendor_name = 'TIVO'
    LEFT JOIN stage.detection.viewing_content_firehose vc
      ON vc.airdate = sch.airdate
    AND sh.show_id = vc.fk_show_id
    AND st.station_id = vc.fk_station_id
    AND vc.session_start >= CURRENT_DATE - 7
    AND vc.session_duration > 0
    GROUP BY 1,2,3
  )
  WHERE session_count = 0
  GROUP BY 1
  ORDER BY 3 DESC, 2 DESC, 1
)
, all_airings AS (
  SELECT station_call_sign, COUNT(DISTINCT database_key) AS total_shows, COUNT(DISTINCT database_key||'_'||airdate) AS total_airings
  FROM dev.mohit_gangwani.fist_2k_unique_ingested_shows
  GROUP BY 1
  ORDER BY 3 DESC, 2 DESC
)
SELECT a.station_call_sign, a.total_airings, n.missed_airings, n.missed_airings*100.0/a.total_airings AS perc_missed
FROM all_airings a
LEFT JOIN no_viewing_airings n
  ON a.station_call_sign = n.station_call_sign
ORDER BY 4

1. find out - out of all the missed airings, how many were captured by any other station
2. Find out how much viewership attribution improved for shows that should have been going to these stations did and did not go to NYC stations

In [0]:
SELECT station_call_sign, COUNT(*)
FROM dev.mohit_gangwani.fist_2k_unique_ingested_shows
GROUP BY 1

In [0]:
SELECT * FROM dev.mohit_gangwani.fist_2k_unique_ingested_shows
WHERE station_call_sign IN ('KKACDT6')

In [0]:
SELECT COUNT(*), COUNT(DISTINCT station_call_sign) FROM dev.mohit_gangwani.ingested_stations_schedule_091525;

In [0]:
SELECT * FROM prod.detection.epg_station
WHERE local_or_national = 'National'
AND (ingested = 'TRUE' OR attributed = 'TRUE')

In [0]:
select vc.* from cure_deltashare.deltasharingtest.qa_r48110_content_with_null vc 
join qa.customer_reports.hashes_qa_deltasharingtest h on h.hash = vc.hash
where ts_start = '2025-09-16 01:29:05' AND ts_end = '2025-09-16 01:30:20'

In [0]:
select vc.*
from cure_deltashare.deltasharingtest.qa_r48110_content_with_null vc 
join qa.customer_reports.hashes_qa_deltasharingtest h on h.hash = vc.hash
WHERE h.tvid = '154264673' AND ts_start = '2025-09-16 01:29:05' AND ts_end = '2025-09-13 01:30:20'

In [0]:
SELECT COUNT(*)
FROM (
SELECT tvid, COUNT(DISTINCT hash) AS ttl_hash
FROM prod.customer_reports.hashes_production_discovery
GROUP BY 1)
WHERE ttl_hash > 1
LIMIT 10

In [0]:
SELECT COUNT(*)
FROM (
SELECT hash, COUNT(DISTINCT tvid) AS ttl_tvid
FROM prod.customer_reports.hashes_production_discovery
GROUP BY 1)
WHERE ttl_tvid > 1
LIMIT 10

In [0]:
SELECT * FROM prod.detection.commercial_id_external_firehose
WHERE external_id IN ('AE16546-2025-21-04569', 'AE16546-2025-25-10717')

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.local_stations_dmas_with_new_stations;
CREATE TABLE dev.mohit_gangwani.local_stations_dmas_with_new_stations AS
SELECT *
FROM stage.detection.epg_station st
WHERE st.vendor_name = 'TIVO'
  AND st.local_or_national = 'Local'
  AND (st.attributed = 'TRUE' OR st.ingested = 'TRUE')
  AND st.fk_dma_id IN (SELECT DISTINCT fk_dma_id FROM dev.mohit_gangwani.2k_first_enabled_set)
  AND st.station_call_sign NOT IN (SELECT DISTINCT station_call_sign FROM dev.mohit_gangwani.2k_first_enabled_set)

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.local_stations_dmas_with_no_new_stations;
CREATE TABLE dev.mohit_gangwani.local_stations_dmas_with_no_new_stations AS
SELECT *
FROM stage.detection.epg_station st
WHERE st.vendor_name = 'TIVO'
  AND st.local_or_national = 'Local'
  AND (st.attributed = 'TRUE' OR st.ingested = 'TRUE')
  AND st.fk_dma_id NOT IN (SELECT DISTINCT fk_dma_id FROM dev.mohit_gangwani.2k_first_enabled_set)

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.national_not_innyc_stations;
CREATE TABLE dev.mohit_gangwani.national_not_innyc_stations AS
SELECT *
FROM stage.detection.epg_station st
WHERE station_call_sign NOT IN ('KJWPDT', 'WABCDT', 'WABCDT2', 'WCBSDT', 'WCBSDT2', 'WCBSDT3', 'WCBSDT4', 'WFUTDT', 'WFUTDT2', 'WFUTDT3', 'WGENDT', 'WJLPDT', 'WJLPDT2', 'WJLPDT3', 'WJLPDT4', 'WLNYDT2', 'WMCNDT2', 'WNBCDT', 'WNBCDT2', 'WNETDT', 'WNJUDT', 'WNYWDT', 'WNYWDT2', 'WNYWDT5', 'WPIXDT', 'WPIXDT2', 'WPIXDT3', 'WPXNDT', 'WPXNDT4', 'WPXNDT6', 'WWORDT', 'WXTVDT', 'WXTVDT2')
  AND st.vendor_name = 'TIVO'
  AND st.local_or_national = 'National'
  AND (st.attributed = 'TRUE' OR st.ingested = 'TRUE');

In [0]:
SELECT *
FROM dev.mohit_gangwani.2k_first_enabled_set k2
JOIN dev.mohit_gangwani.local_stations_dmas_with_new_stations non
  ON k2.station_call_sign = non.station_call_sign


In [0]:
SELECT nat.*
FROM dev.mohit_gangwani.national_not_innyc_stations nat
JOIN dev.mohit_gangwani.ny_nodma_stations nyc
  ON nat.station_call_sign = nyc.station_call_sign  

In [0]:
WITH comm AS (
SELECT fk_commercial_id, external_id FROM prod.detection.commercial_id_external_firehose
WHERE fk_client_id = 753
)
SELECT comm_count, COUNT(*)
FROM (
SELECT fk_tvid, session_start, session_end, COUNT(DISTINCT external_id) AS comm_count
FROM prod.detection.viewing_commercials_firehose vc
JOIN comm
  ON comm.fk_commercial_id = vc.fk_commercial_id
WHERE vc.session_start >= CURRENT_DATE - 1
  AND vc.session_start < CURRENT_DATE
GROUP BY 1, 2, 3)
WHERE comm_count > 1
GROUP BY 1

In [0]:
WITH station_stats AS (
  SELECT DATE_TRUNC('DAY', vc.session_start) AS session_hour
  , vc.fk_dma_id
  , COUNT(*) AS session_count
  , COUNT(DISTINCT vc.fk_tvid) AS tv_count
  , SUM(vc.session_duration)/3600.0 AS total_duration
  FROM stage.detection.viewing_content_firehose vc
  JOIN dev.mohit_gangwani.ny_nodma_stations st
    ON st.station_id = vc.fk_station_id
  WHERE vc.session_start >= '2025-08-15 00:00:00'
  GROUP BY 1, 2
)
SELECT s.session_hour
, CASE WHEN s.fk_dma_id IN (SELECT DISTINCT fk_dma_id FROM dev.mohit_gangwani.2k_first_enabled_set) THEN 'In New Stations DMA'
       ELSE 'Not in New Stations DMA' END AS dma_type
, SUM(s.total_duration) AS station_total_duration
FROM station_stats AS s
GROUP BY 1, 2

In [0]:
SELECT COUNT(DISTINCT series_aggregate_title)
FROM prod.detection.vizio_epg_program_aggregate
WHERE series_aggregate_title LIKE '%,%'

In [0]:
SELECT COUNT(DISTINCT title)
FROM prod.detection.epg_show
WHERE title LIKE '%"%'

In [0]:
SELECT REGEXP_REPLACE('abc,knakfn, kfnkf, kfnakf,fknsfka', ', ?', '|')

In [0]:
SELECT REGEXP_REPLACE('[abc,knakfn, kfnkf, kfnakf,fknsfka]', '[\\[\\]]', '')